# Lesson 10 - ఉత్పత్తిలో AI ఏజెంట్లు

ఈ పాఠంలో మీరు Microsoft Agent Framework (Python) ఉపయోగించి AI ఏజెంట్ల కోసం **ఉత్పత్తి నమూనాలు** నేర్చుకుంటారు. మేము కవర్ చేస్తాము:

- **ఆబ్జర్వబిలిటీ** — ఏజెంట్ ఇంటరాక్షన్లకు టైమింగ్ మరియు లాగింగ్ జోడించడం
- **విలువయిర్చడం** — ప్రతిస్పందన నాణ్యతను స్కోర్ చేయడానికి ఎవాల్యుయేటర్ ఏజెంట్‌ని ఉపయోగించడం
- **ఖర్చు నిర్వహణ** — టోకెన్ ఆప్టిమైజేషన్ మరియు మోడల్ ఎంపిక కోసం వ్యూహాలు

స్నారియో ఒక **ప్రయాణ ఏజెంట్** ఇది వినియోగదారులకు ప్రయాణాలు ఏర్పాటుచేయడంలో సహాయపడుతుంది, దాని పై మానిటరింగ్ మరియు విలువయిర్చడం లేయర్ చేయబడింది.


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ఉత్పత్తి పరిగణనలు

ఏఐ ఏజెంట్లను నమూనాల నుండి ఉత్పత్తికి మార్చేందుకు మూడు మూలస్తంభాలపై జాగ్రత్తగా దృష్టి పెట్టాలి:

1. **నిరీక్షణా శక్తి** — ఏజెంట్ ఏమి చేస్తున్నాడు, ఎంత సేపు తీసుకుంటున్నాడు, ఏ పరికరాలను పిలుస్తున్నాడు అన్నదానిపై మీరు కనిపించేలా ఉండాలి. ట్రేసింగ్ మరియు లాగింగ్ లేకుండా, ఉత్పత్తి సమస్యలను డీబగ్ చేయడం అసాధ్యమే.

2. **మూల్యాంకనం** — ఆటోమేటెడ్ నాణ్యత తనిఖీలు ఏజెంట్ యొక్క ప్రతిస్పందనలు సమయానుకూలంగా సరిగ్గా, పూర్తిగా, మరియు సహాయకంగా ఉంటాయని నిర్దారిస్తాయి. ఒక మూల్యాంకన ఏజెంట్ నిర్వచించబడిన ప్రమాణాలతో ప్రతిస్పందనలు స్కోర్ చేయవచ్చు.

3. **ఖర్చు నిర్వహణ** — టోకెన్ వినియోగం ఖర్చును నేరుగా ప్రభావితం చేస్తుంది. ప్రాంప్ట్ ఆప్టిమైజేషన్, మోడల్ ఎంపిక, మరియు కెషింగ్ వంటి విధానాలు నాణ్యతను తగ్గించకుండా వ్యయాలను నియంత్రణలో పెట్టడంలో సహాయపడతాయి.


## ఒక నిఘా ఎజెంట్ నిర్మించడం

మేము ప్రయాణ సాధనాలను నిర్వచించి, లేటెన్సీని పర్యవేక్షించడానికి టైమింగ్‌తో ఎజెంట్ కాల్‌ను ర్యాప్ చేస్తాము. ఉత్పత్తిలో మీరు OpenTelemetry లేదా సమానమైన ట్రేసింగ్ బ్యాక్‌ఎండ్‌తో ఇంటిగ్రేట్ చేస్తారు.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## మూల్యాంకన నమూనాలు

సాధారణ ఉత్పత్తి నమూనా రెండవ ఏజెంట్‌ను **మూల్యాంకకుడు**గా ఉపయోగించడం. మూల్యాంకకుడు ప్రാഥమిక ఏజెంట్ ప్రతిస్పందనను పూర్తి స్థాయితనం, ఖచ్చితత్వం, మరియు సహాయకారితం వంటి ముందుగా నిర్వచించబడిన ప్రమాణాలలో స్కోరు చేస్తాడు.

ఇది సక్రమంగా చేయడానికి సహాయపడుతుంది:
- ప్రత్యుత్తరాలు వినియోగదారులకు చేరేముందు ఆటోమేటెడ్ నాణ్యత గేట్లు
- ప్రాంప్ట్లు లేదా మోడల్స్ మార్చినప్పుడల్లా తిరుగు ప్రభావం గుర్తింపు
- ఏజెంట్ పని పనితీరు సమయానుసారం నిరంతర పరిణామం నిర్వహణ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ఖర్చు నిర్వహణ వ్యూహాలు

ఉత్పత్తి AI ఏజెంట్ల కోసం ఖర్చులను నియంత్రించడం కీలకం. ఇక్కడ ముఖ్యమైన వ్యూహాలు ఉన్నాయి:

| వ్యూహం | వివరణ |
|---|---|
| **ప్రాంప్ట్ ఆప్టిమైజేషన్** | సిస్టమ్ సూచనలను సంక్షిప్తంగా ఉంచండి. ఇన్‌పుట్ టోకెన్లను తగ్గించడానికి అధికార్థక సందర్భాన్ని తీసివేయండి. |
| **మోడల్ ఎంపిక** | సాదా పనులు (వర్గీకరణ లేదా ఎగ్జ్ట్రాక్షన్ వంటి) కోసం చిన్న, తక్కువ ధరబడిన మోడల్స్ (ఉదాహరణకు GPT-4o-mini) ఉపయోగించండి, మరియు సంక్లిష్ట ఆలోచనలకి పెద్ద మోడల్స్‌ను రిజర్వ్ చేయండి. |
| **కాచింగ్** | పరికర ఫలితాలు మరియు తరచూ అడిగే ప్రశ్నలను కాచింగ్ చేయండి, ఈ విధంగా అనవసర API కాల్‌లను నివారించవచ్చు. |
| **టోకెన్ బడ్జెట్లు** | ఆకస్మికంగా ఎక్కువ సమాధానాలు రాకుండా `max_tokens` పరిమితులను సెట్ చేయండి. |
| **బ్యాచింగ్** | సాధ్యమైతే, ఒకే API కాల్‌లో ఒకటి కంటే ఎక్కువ వినియోగదారు ప్రశ్నలను గ్రూప్ చేయండి. |

వ్యవహారంలో, స్థాయి పద్దతి బాగుంటుంది: సూటిగా ఉండే అభ్యర్థనలను వేగవంతమైన, తక్కువ ఖర్చైన మోడల్‌కు రౌట్ చెయ్యండి మరియు కేవలం సంక్లిష్టమైన ప్రశ్నలు మాత్రమే సమర్థవంతమైన మోడల్‌కు ఎస్కలేట్ చేయండి.


## Summary

ఈ పాఠంలో మీరు నేర్చుకున్నది:

1. **ఏజెంట్ పరస్పర చర్యలకు పర్యవేక్షణను జోడించడం** సమయాల మరియు లాగింగ్ తో, ట్రేసింగ్ మరియు మానిటరింగ్ కోసం మౌలికాన్ని ఏర్పరచడం.
2. **ఏజెంట్ సమాధానాలను** ఆటోమేటిగ్గా మూల్యాంకనం చేయడం, పూర్తి స్థితి, ఖచ్చితత్వం, మరియు సహాయకత ను స్కోర్ చేసే మూల్యాంకక ఏజెంట్ ఉపయోగించడం.
3. **ఖర్చులను నిర్వహించడం** ప్రాంప్ట్ ఆప్టిమైజేషన్, నమూనా ఎంపిక, క్యాచింగ్, మరియు టోకెన్ బడ్జెట్ ద్వారా.

ఈ ప్రొడక్షన్ నమూనాలు మీ AI ఏజెంట్లను విశ్వసనీయమైనది, కొలవగలిగే, మరియు వ్యయ ప్రబంధకంగా వృద్ధిచేయడంలో సహాయపడతాయి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
